In [2]:
"""
scorecard_lib.py
----------------
Shared utilities for the PD scorecard project:
  - coarse binning of numeric / categorical features
  - Weight-of-Evidence (WOE) and Information Value (IV) computation
  - WOE transformation of new data (train-fitted specs)
  - scorecard points scaling

Portfolio project by Vikram Singh, FRM. Synthetic data only.
"""

import numpy as np
import pandas as pd


def _safe_woe(good_counts, bad_counts):
    """WOE with a Laplace-style adjustment so empty bins stay finite."""
    g_total, b_total = good_counts.sum(), bad_counts.sum()
    n = len(good_counts)
    dist_good = (good_counts + 0.5) / (g_total + 0.5 * n)
    dist_bad = (bad_counts + 0.5) / (b_total + 0.5 * n)
    woe = np.log(dist_good / dist_bad)
    iv = (dist_good - dist_bad) * woe
    return woe, iv, dist_good, dist_bad


def fit_woe(x, y, categorical=False, max_bins=10):
    """
    Fit WOE spec for one feature on (x, y).
 
    Returns dict:
      {categorical: bool, bins: list|None, mapping: {bin_label: woe}, iv: float}
    """
    x = pd.Series(x).reset_index(drop=True)
    y = pd.Series(y).reset_index(drop=True)

    if categorical:
        labels = x.astype(str)
        tab = pd.DataFrame({"bin": labels, "bad": y}).groupby("bin")["bad"] \
                .agg(["count", "sum"]).reset_index()
        tab.columns = ["bin", "count", "bad"]
    else:
        edges = np.unique(np.quantile(x, np.linspace(0, 1, max_bins + 1)))
        # need at least 2 distinct edges to form a bin
        if len(edges) < 3:
            edges = np.array([x.min() - 1, x.max() + 1])
        bins = [-np.inf] + list(edges[1:-1]) + [np.inf]
        labels = pd.cut(x, bins=bins, include_lowest=True).astype(str)
        tab = pd.DataFrame({"bin": labels, "bad": y}).groupby("bin")["bad"] \
                .agg(["count", "sum"]).reset_index()
        tab.columns = ["bin", "count", "bad"]

    tab["good"] = tab["count"] - tab["bad"]
    woe, iv, _, _ = _safe_woe(tab["good"].values, tab["bad"].values)
    tab["woe"] = woe
    tab["iv"] = iv

    spec = {
        "categorical": categorical,
        "bins": None if categorical else bins,
        "mapping": dict(zip(tab["bin"], tab["woe"])),
        "iv": float(iv.sum()),
        "table": tab,
    }
    return spec


def transform_woe(x, spec):
    """Apply a fitted WOE spec to new data. Unseen categories -> WOE 0 (neutral)."""
    x = pd.Series(x)
    if spec["categorical"]:
        labels = x.astype(str)
    else:
        labels = pd.cut(x, bins=spec["bins"], include_lowest=True).astype(str)
    woe = labels.map(spec["mapping"]).astype(float).fillna(0.0)
    return woe


def iv_strength(iv):
    """Standard IV interpretive bands."""
    if iv < 0.02:
        return "unpredictive"
    if iv < 0.1:
        return "weak"
    if iv < 0.3:
        return "medium"
    return "strong"


def scorecard_points(feature_woe, beta, intercept, n_features, pdo=40.0, base_score=600.0, base_odds=30.0):
    """
    Classic scorecard scaling (points to double the odds).

    Score = Offset + Factor * ln(odds_good)
      Factor  = PDO / ln(2)
      Offset  = base_score - Factor * ln(base_odds)
    Points contributed by attribute j of feature i:
      -(beta_i * woe_ij + intercept / n_features)
    """
    factor = pdo / np.log(2.0)
    offset = base_score - factor * np.log(base_odds)
    points = -(feature_woe * beta + intercept / n_features)
    return points, factor, offset


In [1]:
"""
01_generate_portfolio.py
------------------------
Generate a synthetic retail/SME loan portfolio for PD scorecard development.

Two samples are produced:
  - Development sample (recent vintages, in-the-money)
  - Out-of-time (OOT) sample with mild distribution drift
    (bureau score and income) to exercise stability testing.

All data is synthetic - generated from a known data-generating process -
so model results can be compared against the "truth". 12-month default
horizon, point-in-time observation.

Run:  python 01_generate_portfolio.py
"""


import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

N_DEV = 25_000
N_OOT = 6_000

PRODUCTS = ["personal", "auto", "mortgage", "SME"]
EMPLOYMENT = ["salaried", "self_employed", "professional"]
REGIONS = ["metro", "tier1", "tier2", "tier3"]

def generate(n, oot=False):
    """Synthetic origination-level data. `oot=True` applies mild drift."""
    age = np.clip(rng.normal(38, 10, n), 21, 65).round(0)

    log_income = rng.normal(12.8, 0.45, n)
    if oot:  # income stress in later vintages
        log_income = log_income - 0.06
    annual_income = np.exp(log_income).round(-3)

    bureau_mean = 700 if not oot else 690  # bureau quality drifts down
    bureau_score = np.clip(rng.normal(bureau_mean, 80, n), 300, 900).round(0)

    product = rng.choice(PRODUCTS, n, p=[0.35, 0.20, 0.20, 0.25])
    employment = rng.choice(EMPLOYMENT, n, p=[0.55, 0.30, 0.15])
    region = rng.choice(REGIONS, n, p=[0.4, 0.25, 0.2, 0.15])

    # loan size scales with income; mortgages and SME are larger tickets
    size_mult = np.where(product == "mortgage", 4.0,
                np.where(product == "SME", 2.5, 1.0))
    loan_amount = (annual_income * size_mult * rng.uniform(0.5, 1.5, n)).round(-4)
    tenure_months = np.where(product == "mortgage", rng.integers(120, 300, n),
                      np.where(product == "auto", rng.integers(36, 84, n),
                      np.where(product == "SME", rng.integers(48, 120, n),
                               rng.integers(12, 60, n))))

    emi = loan_amount / tenure_months * 1.4  # rough annuity at ~14% p.a.
    other_emi = annual_income / 12 * rng.uniform(0.05, 0.35, n)
    dti_ratio = np.clip((emi + other_emi) / (annual_income / 12), 0, 3).round(3)

    delinq_24m = rng.poisson(0.25, n)
    months_since_delinq = np.where(delinq_24m > 0,
                                   rng.integers(1, 24, n), 99)

    # ---- latent default data-generating process (12-month PD) ----
    logit = (-3.90
             + 0.95 * (680 - bureau_score) / 80
             + 1.30 * (dti_ratio - 0.45)
             - 0.70 * (log_income - 12.8) / 0.45
             + 0.45 * (employment == "self_employed")
             + 0.40 * (product == "SME")
             - 0.30 * (product == "mortgage")
             + 0.40 * np.minimum(delinq_24m, 3)
             + 0.25 * (region == "tier3"))
    pd_true = 1 / (1 + np.exp(-logit))
    default_12m = rng.binomial(1, pd_true)

    df = pd.DataFrame({
        "age": age,
        "annual_income": annual_income,
        "bureau_score": bureau_score,
        "product_type": product,
        "employment_type": employment,
        "region": region,
        "loan_amount": loan_amount,
        "tenure_months": tenure_months,
        "dti_ratio": dti_ratio,
        "delinq_24m": delinq_24m,
        "months_since_delinq": months_since_delinq,
        "pd_true": pd_true.round(5),
        "default_12m": default_12m,
        # outstanding balance as EAD proxy (70-100% of origination)
        "ead": (loan_amount * rng.uniform(0.70, 1.0, n)).round(-3),
        "sample": "oot" if oot else "dev",
    })
    return df


dev = generate(N_DEV, oot=False)
oot = generate(N_OOT, oot=True)

dev.to_csv("data/portfolio_dev.csv", index=False)
oot.to_csv("data/portfolio_oot.csv", index=False)

print(f"Development sample : {len(dev):,} rows | default rate {dev['default_12m'].mean():.2%}")
print(f"Out-of-time sample : {len(oot):,} rows | default rate {oot['default_12m'].mean():.2%}")
print("\nSegment default rates (dev):")
print(dev.groupby("product_type")["default_12m"].agg(["count", "mean"]).round(4))


Development sample : 25,000 rows | default rate 5.49%
Out-of-time sample : 6,000 rows | default rate 7.02%

Segment default rates (dev):
              count    mean
product_type               
SME            6248  0.0775
auto           4987  0.0419
mortgage       5047  0.0321
personal       8718  0.0594


In [3]:
"""
02_woe_iv_segmentation.py
-------------------------
Exploratory risk analysis, univariate WOE/IV screening, and the
segmentation assessment for the PD scorecard.

Outputs:
  outputs/woe_iv_summary.csv   - IV per candidate feature
  outputs/woe_bureau_score.csv - WOE table for the strongest feature
  figures/segment_default_rates.png
  figures/woe_bureau_score.png

Run:  python 02_woe_iv_segmentation.py
"""

%pip install -q matplotlib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scorecard_lib import fit_woe, iv_strength

dev = pd.read_csv("data/portfolio_dev.csv")

NUMERIC = ["age", "annual_income", "bureau_score", "loan_amount",
           "tenure_months", "dti_ratio", "delinq_24m", "months_since_delinq"]
CATEGORICAL = ["product_type", "employment_type", "region"]

TARGET = "default_12m"

# ---------- 1. Univariate WOE / IV screening ----------
rows = []
for col in NUMERIC:
    spec = fit_woe(dev[col], dev[TARGET], categorical=False, max_bins=10)
    rows.append({"feature": col, "type": "numeric", "iv": round(spec["iv"], 4),
                 "strength": iv_strength(spec["iv"])})
for col in CATEGORICAL:
    spec = fit_woe(dev[col], dev[TARGET], categorical=True)
    rows.append({"feature": col, "type": "categorical", "iv": round(spec["iv"], 4),
                 "strength": iv_strength(spec["iv"])})

iv_summary = pd.DataFrame(rows).sort_values("iv", ascending=False)
iv_summary.to_csv("outputs/woe_iv_summary.csv", index=False)
print("Information Value summary (12-month default):")
print(iv_summary.to_string(index=False))

# ---------- 2. WOE table for the strongest numeric driver ----------
bureau_spec = fit_woe(dev["bureau_score"], dev[TARGET], categorical=False)
tab = bureau_spec["table"][["bin", "count", "good", "bad", "woe", "iv"]]
tab.to_csv("outputs/woe_bureau_score.csv", index=False)
print("\nWOE - bureau_score:")
print(tab.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
labels = [l.split(",")[0].strip("(") for l in tab["bin"]]
ax.bar(range(len(tab)), tab["woe"], color="#1F3864")
ax.set_xticks(range(len(tab)))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Bureau score (bin lower edge)")
ax.set_ylabel("WOE")
ax.set_title("Weight of Evidence by bureau score bin")
fig.tight_layout()
fig.savefig("figures/woe_bureau_score.png", dpi=150)

# ---------- 3. Segmentation assessment ----------
seg = dev.groupby("product_type").agg(
    n=(TARGET, "size"),
    default_rate=(TARGET, "mean"),
    avg_ead=("ead", "mean"),
).round(4)
print("\nSegment profile:")
print(seg.to_string())

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(seg.index, seg["default_rate"], color="#1F3864")
ax.set_ylabel("12-month default rate")
ax.set_title("Default rate by product segment (development sample)")
for i, (idx, r) in enumerate(seg.iterrows()):
    ax.text(i, r["default_rate"] + 0.001, f"{r['default_rate']:.2%}\nn={int(r['n']):,}",
            ha="center", fontsize=9)
fig.tight_layout()
fig.savefig("figures/segment_default_rates.png", dpi=150)

# Monotonic risk ordering check on bureau score
print("\nBinned default rate vs bureau score (monotonicity check):")
brk = [-np.inf, 480, 540, 600, 660, 720, 780, np.inf]
check = dev.groupby(pd.cut(dev["bureau_score"], brk), observed=False)[TARGET].agg(["count", "mean"])
print(check.round(4).to_string())

print("\nNotes for segmentation decision:")
print("- SME default rate materially above other products; mortgage below.")
print("- Product type will enter the pooled model as a WOE-encoded feature;")
print("  a dedicated segmented-vs-pooled performance comparison is run in")
print("  03_pd_scorecard_development.py before finalising the structure.")


Note: you may need to restart the kernel to use updated packages.
Information Value summary (12-month default):
            feature        type     iv     strength
       bureau_score     numeric 0.7369       strong
      annual_income     numeric 0.3820       strong
          dti_ratio     numeric 0.1382       medium
        loan_amount     numeric 0.1186       medium
       product_type categorical 0.1060       medium
      tenure_months     numeric 0.0880         weak
    employment_type categorical 0.0524         weak
         delinq_24m     numeric 0.0277         weak
months_since_delinq     numeric 0.0270         weak
             region categorical 0.0172 unpredictive
                age     numeric 0.0101 unpredictive

WOE - bureau_score:
           bin  count  good  bad       woe       iv
 (-inf, 597.0]   2524  2096  428 -1.254235 0.278745
(597.0, 633.0]   2535  2273  262 -0.683144 0.064413
(633.0, 658.0]   2542  2368  174 -0.233881 0.006176
(658.0, 680.0]   2501  2361  140 -0

In [6]:
"""
03_pd_scorecard_development.py
------------------------------
Develop the PD scorecard:

  1. Split the development sample (70/30, stratified).
  2. Fit WOE binning specs on the training partition ONLY
     (no leakage of out-of-sample information into transformations).
  3. WOE-encoded logistic regression -> 12-month PD.
  4. Compare pooled model vs per-segment models (segmentation decision).
  5. Scale to a credit scorecard (points-to-double-odds).
  6. Score dev-train, dev-test and the out-of-time sample.

Outputs:
  outputs/scorecard_points.csv   - human-readable scorecard (attribute points)
  outputs/model_coefficients.csv
  data/scored_portfolio.csv      - all rows with pd_hat, score, sample tag
  figures/roc_test.png, figures/score_distribution.png

Run:  python 03_pd_scorecard_development.py
"""

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from scorecard_lib import fit_woe, transform_woe, scorecard_points

TARGET = "default_12m"
PDO, BASE_SCORE, BASE_ODDS = 40.0, 600.0, 30.0

NUMERIC = ["age", "annual_income", "bureau_score", "loan_amount",
           "tenure_months", "dti_ratio", "delinq_24m"]
CATEGORICAL = ["product_type", "employment_type", "region"]
FEATURES = NUMERIC + CATEGORICAL

dev = pd.read_csv("data/portfolio_dev.csv")
oot = pd.read_csv("data/portfolio_oot.csv")

# ---------- 1. train / test split ----------
train, test = train_test_split(dev, test_size=0.30, stratify=dev[TARGET], random_state=7)
train, test = train.copy(), test.copy()
train["sample"], test["sample"] = "dev_train", "test"
print(f"Train: {len(train):,} (DR {train[TARGET].mean():.2%}) | "
      f"Test: {len(test):,} (DR {test[TARGET].mean():.2%})")

# ---------- 2. WOE specs fitted on TRAIN only ----------
specs = {}
for col in NUMERIC:
    specs[col] = fit_woe(train[col], train[TARGET], categorical=False, max_bins=10)
for col in CATEGORICAL:
    specs[col] = fit_woe(train[col], train[TARGET], categorical=True)


def design_matrix(df):
    X = pd.DataFrame({col: transform_woe(df[col], specs[col]) for col in FEATURES})
    return X


X_train, X_test = design_matrix(train), design_matrix(test)
y_train, y_test = train[TARGET], test[TARGET]

# ---------- 3. logistic regression on WOE features ----------
model = LogisticRegression(C=np.inf, solver="lbfgs", max_iter=1000)
model.fit(X_train, y_train)

auc_test = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f"\nPooled model  | test AUC = {auc_test:.4f}  (Gini = {2*auc_test-1:.4f})")

# ---------- 4. pooled vs segmented models ----------
print("\nSegmentation check - AUC of pooled vs segment-specific models (test):")
seg_defs = {
    "mortgage": test["product_type"] == "mortgage",
    "SME": test["product_type"] == "SME",
    "retail_other": ~test["product_type"].isin(["mortgage", "SME"]),
}
pooled_prob = model.predict_proba(X_test)[:, 1]
best_lift = 0.0
for seg_name, mask in seg_defs.items():
    y_seg, p_seg = y_test[mask.values], pooled_prob[mask.values]
    pooled_auc = roc_auc_score(y_seg, p_seg)

    tr_mask = (train["product_type"] == seg_name.replace("retail_other", "XX")) \
        if seg_name != "retail_other" else ~train["product_type"].isin(["mortgage", "SME"])
    m = LogisticRegression(C=np.inf, solver="lbfgs", max_iter=1000)
    m.fit(X_train[tr_mask.values], y_train[tr_mask.values])
    p_hat = m.predict_proba(X_test[mask.values])[:, 1]
    seg_auc = roc_auc_score(y_seg, p_hat)

    lift = seg_auc - pooled_auc
    best_lift = max(best_lift, lift)
    print(f"  {seg_name:<13} pooled AUC {pooled_auc:.4f} | segmented AUC {seg_auc:.4f} | "
          f"lift {lift:+.4f}")

if best_lift < 0.01:
    print("\nDECISION: segmentation lift < 1 AUC point in every segment ->")
    print("retain the POOLED model with product_type as a WOE-encoded driver")
    print("(simpler to govern, calibrate and document under model risk policy).")
else:
    print("\nDECISION: material lift in at least one segment -> consider")
    print("segmented scorecards subject to minimum segment size checks.")

# ---------- 5. scorecard scaling ----------
coefs = dict(zip(FEATURES, model.coef_[0]))
intercept = model.intercept_[0]
pd.DataFrame({"feature": FEATURES, "beta_woe": [coefs[c] for c in FEATURES]}) \
    .to_csv("outputs/model_coefficients.csv", index=False)

points_frames = []
for col in FEATURES:
    spec = specs[col]
    t = spec["table"]
    pts, factor, offset = scorecard_points(
        t["woe"].values, coefs[col], intercept, len(FEATURES),
        pdo=PDO, base_score=BASE_SCORE, base_odds=BASE_ODDS)
    points_frames.append(pd.DataFrame({
        "feature": col, "attribute": t["bin"],
        "count": t["count"], "woe": t["woe"].round(4),
        "points": pts.round(1),
    }))
scorecard = pd.concat(points_frames, ignore_index=True)
scorecard.to_csv("outputs/scorecard_points.csv", index=False)
print(f"\nScorecard scaling: PDO={PDO:.0f}, {BASE_SCORE:.0f} points at "
      f"good:bad odds {BASE_ODDS:.0f}:1  (factor {factor:.2f}, offset {offset:.1f})")

# ---------- 6. score all samples ----------
for name, df in [("train", train), ("test", test), ("oot", oot)]:
    X = design_matrix(df)
    logit = intercept + sum(coefs[c] * X[c] for c in FEATURES)
    df["pd_hat"] = 1 / (1 + np.exp(-logit))
    df["score"] = offset + factor * np.log((1 - df["pd_hat"]) / df["pd_hat"])

scored = pd.concat([train, test, oot], ignore_index=True)
scored.to_csv("data/scored_portfolio.csv", index=False)

# ---------- figures ----------
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_test, pooled_prob)
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot(fpr, tpr, color="#1F3864", label=f"WOE logistic PD (AUC {auc_test:.3f})")
ax.plot([0, 1], [0, 1], "--", color="grey", linewidth=1)
ax.set_xlabel("False positive rate"), ax.set_ylabel("True positive rate")
ax.set_title("ROC - holdout test sample"), ax.legend()
fig.tight_layout(); fig.savefig("figures/roc_test.png", dpi=150)

fig, ax = plt.subplots(figsize=(7, 4))
for flag, color, lbl in [(0, "#2E7D32", "Good (no default)"), (1, "#C62828", "Bad (default)")]:
    ax.hist(scored[(scored["sample"] != "oot") & (scored[TARGET] == flag)]["score"],
            bins=60, alpha=0.55, color=color, label=lbl, density=True)
ax.set_xlabel("Credit score"), ax.set_ylabel("Density")
ax.set_title("Score distribution - development sample")
ax.legend()
fig.tight_layout(); fig.savefig("figures/score_distribution.png", dpi=150)

print("\nSaved: outputs/scorecard_points.csv, outputs/model_coefficients.csv,")
print("data/scored_portfolio.csv, figures/roc_test.png, figures/score_distribution.png")


Train: 17,500 (DR 5.49%) | Test: 7,500 (DR 5.49%)

Pooled model  | test AUC = 0.7909  (Gini = 0.5818)

Segmentation check - AUC of pooled vs segment-specific models (test):
  mortgage      pooled AUC 0.7326 | segmented AUC 0.7280 | lift -0.0046
  SME           pooled AUC 0.7818 | segmented AUC 0.7812 | lift -0.0006
  retail_other  pooled AUC 0.7956 | segmented AUC 0.7948 | lift -0.0008

DECISION: segmentation lift < 1 AUC point in every segment ->
retain the POOLED model with product_type as a WOE-encoded driver
(simpler to govern, calibrate and document under model risk policy).

Scorecard scaling: PDO=40, 600 points at good:bad odds 30:1  (factor 57.71, offset 403.7)

Saved: outputs/scorecard_points.csv, outputs/model_coefficients.csv,
data/scored_portfolio.csv, figures/roc_test.png, figures/score_distribution.png


In [8]:
"""
04_model_validation.py
----------------------
Independent validation of the PD scorecard, in the style of a second-line
model-validation function:

  Discrimination  : AUC, Gini, Kolmogorov-Smirnov (KS)
  Calibration     : decile predicted-vs-observed, Hosmer-Lemeshow test, Brier
  Classification  : confusion matrix at the score cut-off
  Stability       : Population Stability Index (dev-train vs out-of-time)
                    on both the bureau score and the model score

Outputs:
  outputs/validation_summary.csv
  outputs/calibration_deciles.csv
  figures/calibration.png, figures/ks_curve.png, figures/stability_psi.png

Run:  python 04_model_validation.py
"""

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import chi2
from sklearn.metrics import roc_auc_score, brier_score_loss, confusion_matrix

TARGET = "default_12m"

df = pd.read_csv("data/scored_portfolio.csv")
test = df[df["sample"] == "test"]
train = df[df["sample"] == "dev_train"]
oot = df[df["sample"] == "oot"]

# ---------- Discrimination ----------
auc = roc_auc_score(test[TARGET], test["pd_hat"])
gini = 2 * auc - 1
scores_sorted = np.sort(test["score"].values)
s_bad = np.sort(test.loc[test[TARGET] == 1, "score"])
s_good = np.sort(test.loc[test[TARGET] == 0, "score"])
ecdf_bad = np.searchsorted(s_bad, scores_sorted, side="right") / len(s_bad)
ecdf_good = np.searchsorted(s_good, scores_sorted, side="right") / len(s_good)
ks = np.max(np.abs(ecdf_bad - ecdf_good))

# ---------- Calibration ----------
test2 = test.copy()
test2["decile"] = pd.qcut(test2["pd_hat"].rank(method="first"), 10, labels=False) + 1
cal = test2.groupby("decile").agg(
    n=(TARGET, "size"),
    predicted_pd=("pd_hat", "mean"),
    observed_dr=(TARGET, "mean"),
).reset_index()
cal["abs_gap_pp"] = ((cal["predicted_pd"] - cal["observed_dr"]) * 100).abs().round(2)
cal.to_csv("outputs/calibration_deciles.csv", index=False)

# Hosmer-Lemeshow statistic (10 groups)
hl = ((cal["n"] * (cal["predicted_pd"] - cal["observed_dr"]) ** 2) /
      (cal["predicted_pd"] * (1 - cal["predicted_pd"]))).sum()
hl_p = chi2.sf(hl, df=8)
brier = brier_score_loss(test[TARGET], test["pd_hat"])

# ---------- Classification at cut-off (score < 560) ----------
cutoff = 560
test2["flag"] = (test2["score"] < cutoff).astype(int)
cm = confusion_matrix(test2[TARGET], test2["flag"])
tn, fp, fn, tp = cm.ravel()
accept_rate = 1 - test2["flag"].mean()
bad_capture = tp / (tp + fn)

# ---------- Stability: PSI ----------
def psi(expected, actual, bins=10):
    """Population Stability Index between two samples of one variable."""
    edges = np.unique(np.quantile(expected, np.linspace(0, 1, bins + 1)))
    edges = np.concatenate([[-np.inf], edges[1:-1], [np.inf]])
    e = np.histogram(expected, bins=edges)[0] / len(expected)
    a = np.histogram(actual, bins=edges)[0] / len(actual)
    e, a = np.clip(e, 1e-6, None), np.clip(a, 1e-6, None)
    return float(((a - e) * np.log(a / e)).sum())


def psi_label(v):
    return "stable" if v < 0.1 else ("moderate - monitor" if v < 0.25 else "material shift")


psi_bureau = psi(train["bureau_score"], oot["bureau_score"])
psi_score = psi(train["score"], oot["score"])

summary = pd.DataFrame([
    ("AUC (test)", f"{auc:.4f}", ">= 0.70 typical retail objective"),
    ("Gini (test)", f"{gini:.4f}", "2*AUC-1"),
    ("KS statistic (test)", f"{ks:.4f}", ">= 0.30 healthy separation"),
    ("Brier score (test)", f"{brier:.5f}", "lower is better"),
    ("Hosmer-Lemeshow chi2 (8 df)", f"{hl:.2f} (p={hl_p:.3f})",
     "p > 0.05 -> no evidence of miscalibration"),
    ("PSI bureau score (train vs OOT)", f"{psi_bureau:.4f} ({psi_label(psi_bureau)})",
     "<0.10 stable / 0.10-0.25 monitor / >0.25 shift"),
    ("PSI model score (train vs OOT)", f"{psi_score:.4f} ({psi_label(psi_score)})",
     "drift in output distribution"),
    (f"Approval rate @ score>={cutoff}", f"{accept_rate:.2%}", "policy cut-off"),
    (f"Bad-capture @ score<{cutoff}", f"{bad_capture:.2%}",
     "share of eventual defaults below cut-off"),
], columns=["metric", "value", "benchmark / note"])
summary.to_csv("outputs/validation_summary.csv", index=False)
print(summary.to_string(index=False))

# ---------- Figures ----------
fig, ax = plt.subplots(figsize=(5.5, 5))
lim = cal["observed_dr"].max() * 1.1
ax.plot([0, lim], [0, lim], "--", color="grey", linewidth=1)
ax.scatter(cal["observed_dr"], cal["predicted_pd"], color="#1F3864", s=45)
ax.set_xlabel("Observed default rate (decile)")
ax.set_ylabel("Mean predicted PD (decile)")
ax.set_title(f"Calibration - holdout test (HL p = {hl_p:.2f})")
fig.tight_layout(); fig.savefig("figures/calibration.png", dpi=150)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(scores_sorted, ecdf_bad, color="#C62828", label="Bad")
ax.plot(scores_sorted, ecdf_good, color="#2E7D32", label="Good")
ax.set_xlabel("Score"), ax.set_ylabel("Cumulative share")
ax.set_title(f"KS curve (KS = {ks:.3f})"), ax.legend()
fig.tight_layout(); fig.savefig("figures/ks_curve.png", dpi=150)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(train["score"], bins=60, alpha=0.55, density=True, label="Development", color="#1F3864")
ax.hist(oot["score"], bins=60, alpha=0.55, density=True, label="Out-of-time", color="#E8A33D")
ax.set_xlabel("Score"), ax.set_ylabel("Density")
ax.set_title(f"Population stability - score distribution (PSI = {psi_score:.3f})")
ax.legend()
fig.tight_layout(); fig.savefig("figures/stability_psi.png", dpi=150)

print("\nSaved: outputs/validation_summary.csv, outputs/calibration_deciles.csv,")
print("figures/calibration.png, figures/ks_curve.png, figures/stability_psi.png")


                         metric           value                               benchmark / note
                     AUC (test)          0.7909               >= 0.70 typical retail objective
                    Gini (test)          0.5818                                        2*AUC-1
            KS statistic (test)          0.4390                     >= 0.30 healthy separation
             Brier score (test)         0.04765                                lower is better
    Hosmer-Lemeshow chi2 (8 df)  8.15 (p=0.419)      p > 0.05 -> no evidence of miscalibration
PSI bureau score (train vs OOT) 0.0168 (stable) <0.10 stable / 0.10-0.25 monitor / >0.25 shift
 PSI model score (train vs OOT) 0.0292 (stable)                   drift in output distribution
     Approval rate @ score>=560          73.05%                                 policy cut-off
        Bad-capture @ score<560          67.96%       share of eventual defaults below cut-off

Saved: outputs/validation_summary.csv, outputs/ca

In [1]:
"""
05_capital_impact_rwa.py
------------------------
Translate the PD scorecard into risk capital under the Basel
Internal Ratings-Based (IRB) approach, and compare with the
standardized approach (SA).

For each exposure in the scored portfolio:
  PD    : model 12-month PD (floored at 0.03% per supervisory practice)
  LGD   : illustrative F-IRB-style assumptions
            - residential mortgage: 25%
            - all other retail    : 45%
  EAD   : outstanding balance from the synthetic portfolio

Retail IRB risk weight (Basel formula):
  b = (0.11852 - 0.05478 * ln(PD))^2
  K = [LGD * N((N^-1(PD) + sqrt(R) * N^-1(0.999)) / sqrt(1 - R)) - PD * LGD]
        * (1 / (1 - 1.5 * b))
  RWA = 12.5 * K * EAD

  Asset correlations (retail):
    residential mortgage : R = 0.15
    other retail        : R = 0.03 * (1 - e^-35PD)/(1 - e^-35)
                            + 0.16 * (1 - (1 - e^-35PD)/(1 - e^-35))

Standardized comparison (illustrative pre-reform basis):
  residential mortgage RW = 35%, other retail RW = 100%

Also reports Expected Loss:  EL = PD x LGD x EAD.

NOTE: formulas implemented for demonstration on synthetic data; a production
IRB capital engine carries supervisory floors, scaling factors and CRM
treatment that are out of scope here.

Run:  python 05_capital_impact_rwa.py
"""

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm

PD_FLOOR = 0.0003

df = pd.read_csv("data/scored_portfolio.csv")

df["PD"] = df["pd_hat"].clip(lower=PD_FLOOR)
df["LGD"] = np.where(df["product_type"] == "mortgage", 0.25, 0.45)


def retail_correlation(pd_, mortgage):
    """Basel retail asset correlation."""
    other = 0.03 * (1 - np.exp(-35 * pd_)) / (1 - np.exp(-35)) \
          + 0.16 * (1 - (1 - np.exp(-35 * pd_)) / (1 - np.exp(-35)))
    return np.where(mortgage, 0.15, other)


def irb_k(pd_, lgd, mortgage):
    """Retail IRB capital requirement K."""
    R = retail_correlation(pd_, mortgage)
    b = (0.11852 - 0.05478 * np.log(pd_)) ** 2
    cond = norm.cdf((norm.ppf(pd_) + np.sqrt(R) * norm.ppf(0.999)) / np.sqrt(1 - R))
    k = (lgd * cond - pd_ * lgd) / (1 - 1.5 * b)
    return np.maximum(k, 0.0)


df["K"] = irb_k(df["PD"].values, df["LGD"].values, (df["product_type"] == "mortgage").values)
df["RWA_IRB"] = 12.5 * df["K"] * df["ead"]
df["RW_SA"] = np.where(df["product_type"] == "mortgage", 0.35, 1.00)
df["RWA_SA"] = df["RW_SA"] * df["ead"]
df["EL"] = df["PD"] * df["LGD"] * df["ead"]

# ---------- portfolio summary ----------
res = df.groupby("product_type").agg(
    n=("ead", "size"),
    EAD_bn=("ead", lambda s: s.sum() / 1e9),
    mean_PD=("PD", "mean"),
    EL_mn=("EL", lambda s: s.sum() / 1e7),
    RWA_IRB_bn=("RWA_IRB", lambda s: s.sum() / 1e9),
    RWA_SA_bn=("RWA_SA", lambda s: s.sum() / 1e9),
).reset_index()
res["IRB_density_%"] = (res["RWA_IRB_bn"] / res["EAD_bn"] * 100).round(1)
res["SA_density_%"] = (res["RWA_SA_bn"] / res["EAD_bn"] * 100).round(1)
res["capital_saved_%"] = ((1 - res["RWA_IRB_bn"] / res["RWA_SA_bn"]) * 100).round(1)
res = res.round({"mean_PD": 4, "EAD_bn": 2, "EL_mn": 1, "RWA_IRB_bn": 2, "RWA_SA_bn": 2})
res.to_csv("outputs/capital_summary.csv", index=False)
print("Portfolio capital summary (synthetic data, illustrative assumptions):")
print(res.to_string(index=False))

total = res[["EAD_bn", "EL_mn", "RWA_IRB_bn", "RWA_SA_bn"]].sum()
print(f"\nPortfolio total : EAD Rs.{total['EAD_bn']:.2f} bn | "
      f"EL Rs.{total['EL_mn']:.1f} cr | RWA IRB Rs.{total['RWA_IRB_bn']:.2f} bn | "
      f"RWA SA Rs.{total['RWA_SA_bn']:.2f} bn")
print(f"Overall RWA density: IRB {total['RWA_IRB_bn']/total['EAD_bn']:.1%} vs "
      f"SA {total['RWA_SA_bn']/total['EAD_bn']:.1%}")

# ---------- RWA density vs PD curve (mortgage vs other retail) ----------
pds = np.linspace(0.0005, 0.20, 100)
rw_mort = 12.5 * irb_k(pds, 0.25, np.ones_like(pds, dtype=bool))
rw_other = 12.5 * irb_k(pds, 0.45, np.zeros_like(pds, dtype=bool))

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(pds * 100, rw_mort * 100, label="Residential mortgage (LGD 25%, R=15%)", color="#1F3864")
ax.plot(pds * 100, rw_other * 100, label="Other retail (LGD 45%, PD-dependent R)", color="#E8A33D")
ax.axhline(35, ls="--", lw=1, color="#1F3864", label="SA mortgage RW 35%")
ax.axhline(100, ls="--", lw=1, color="#E8A33D", label="SA other retail RW 100%")
ax.set_xlabel("PD (%)"), ax.set_ylabel("Risk weight (%)")
ax.set_title("Basel retail IRB risk weight as a function of PD")
ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig("figures/rw_vs_pd.png", dpi=150)

# ---------- portfolio RWA comparison chart ----------
fig, ax = plt.subplots(figsize=(7, 4.5))
x = np.arange(len(res))
ax.bar(x - 0.2, res["RWA_IRB_bn"], 0.4, label="IRB RWA", color="#1F3864")
ax.bar(x + 0.2, res["RWA_SA_bn"], 0.4, label="Standardized RWA", color="#9AA7B8")
ax.set_xticks(x), ax.set_xticklabels(res["product_type"])
ax.set_ylabel("RWA (Rs bn)"), ax.set_title("RWA by product - IRB vs standardized")
ax.legend()
fig.tight_layout(); fig.savefig("figures/rwa_comparison.png", dpi=150)

print("\nSaved: outputs/capital_summary.csv, figures/rw_vs_pd.png, figures/rwa_comparison.png")


Portfolio capital summary (synthetic data, illustrative assumptions):
product_type     n  EAD_bn  mean_PD  EL_mn  RWA_IRB_bn  RWA_SA_bn  IRB_density_%  SA_density_%  capital_saved_%
         SME  7723    6.45   0.0746   17.6        4.67       6.45           72.4         100.0             27.6
        auto  6172    2.09   0.0431    3.2        1.32       2.09           63.2         100.0             36.8
    mortgage  6251    8.42   0.0306    5.1        4.36       2.95           51.8          35.0            -48.1
    personal 10854    3.64   0.0677    9.0        2.57       3.64           70.6         100.0             29.4

Portfolio total : EAD Rs.20.60 bn | EL Rs.34.9 cr | RWA IRB Rs.12.92 bn | RWA SA Rs.15.13 bn
Overall RWA density: IRB 62.7% vs SA 73.4%

Saved: outputs/capital_summary.csv, figures/rw_vs_pd.png, figures/rwa_comparison.png
